# MBH-TOPTW: caso chileno, versión 2 (superficies oficiales de las AMP)

**Qué cambia respecto de la versión 1.** Las superficies de las AMP oceánicas y austral se toman de los decretos de creación
(redondeadas al millar de km²): Rapa Nui + Motu Motiro Hiva 729.000, Nazca-Desventuradas 300.000, Juan Fernández 286.000
(parque 262.000 + AMCP-MU 24.000, DS 12/2018) y Diego Ramírez-Paso Drake 144.000 (DS 9/2018). El sector "Paso Drake" pasa a
tener su punto de referencia dentro del parque (centroide de la mitad sur, 58,69°S 70,30°W). Todo lo demás (flota, bases,
reglas de probabilidad, puntaje y ventanas, modelos y parámetros de los métodos) es igual al de la campaña del paper.

**Qué hace.** (1) Reconstruye la instancia y enumera todos sus planes óptimos (no usa Gurobi). (2) Corre MILP, CG y CG-TS con
los mismos parámetros de la campaña (límite 3.600 s, pricing exacto 60 s, MIP final 600 s). (3) Arma un zip con todo.
Tarda entre 2 y 5 minutos.

**Cómo se corre.**
1. Licencia Gurobi WLS: igual que la vez anterior (Colab Secrets `WLSACCESSID`, `WLSSECRET`, `LICENSEID`, o el archivo
   `MyDrive/mbh_toptw/gurobi.lic`). El notebook no imprime la licencia.
2. Usa una máquina estándar de Colab (2 CPU, la misma de la campaña) para que los tiempos sean comparables.
3. *Entorno de ejecución → Ejecutar todo*. Al final se descarga `mbh_caso_v2_resultados.zip` (copia en
   `MyDrive/mbh_toptw/caso_v2/`). **Súbelo al chat**; si la descarga falla, pega el texto de la última celda.

## 0. Dependencias y licencia

In [ ]:
!pip -q install "gurobipy>=12" pandas
import os, sys, json, time, shutil, glob
import numpy as np, pandas as pd

LIC_PATH = '/content/gurobi.lic'
try:
    from google.colab import userdata
    acc, sec, lid = userdata.get('WLSACCESSID'), userdata.get('WLSSECRET'), userdata.get('LICENSEID')
    with open(LIC_PATH, 'w') as f:
        f.write(f"WLSACCESSID={acc}\nWLSSECRET={sec}\nLICENSEID={lid}\n")
    print('licencia escrita desde Colab Secrets')
except Exception as e:
    print('Secrets no disponibles (', str(e)[:60], ') -> se usara MyDrive/mbh_toptw/gurobi.lic')
os.environ['GRB_LICENSE_FILE'] = LIC_PATH
os.makedirs('/content/mbh_v2/code', exist_ok=True)

## 1. Código (versión 2 del caso; no hay que modificar nada)
Las celdas siguientes solo escriben los archivos de código en `/content/mbh_v2/code`.

In [ ]:
%%writefile /content/mbh_v2/code/mbh_core.py
"""
mbh_core.py  --  Multi-Base Heterogeneous Team Orienteering Problem with Time Windows (MBH-TOPTW)

Faithful Python port of the Java/CPLEX implementation (Canut-de-Bon, 2019-2026), plus:
  * a route-schedule evaluator with exact minimum-duration computation under time windows,
  * a Tabu Search pricing heuristic (new),
  * column generation with TS pricing certified by an exact MIP pricing (valid Lagrangian bound),
  * a pure-Python LP backend (dense simplex) used only for testing without a commercial solver.

Units (identical to the Java code):
  coordinates in km (Euclidean), speed in km/h, time in MINUTES,
  travel time  t_ij^r  = dist(i,j) * 60 / v_r
  service time tau_i^r = size_i * SIZE_SCALE * 60 / (w_r * v_r)      (SIZE_SCALE = 100: sizes are in 100 km^2)
  a route is feasible if there is a departure time d >= 0 such that every area is served inside its
  window [o_i, c_i] (waiting allowed) and the return to the end base occurs no later than d + T_r.

Node indexing inside this module: areas 0..V-1, bases V..V+D-1 (a base serves both as departure and arrival).
"""
from __future__ import annotations
import math, time, random, os, json
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
import numpy as np

SIZE_SCALE = 100.0
TIME_SCALE = 60.0
INF = float('inf')

# =====================================================================================
#  Instance
# =====================================================================================
@dataclass
class Aircraft:
    T: float            # endurance (min)
    v: float            # speed (km/h)
    w: float            # detection range / sweep width (km)
    start: int          # index of departure base in 0..D-1, or -1 if free

@dataclass
class Instance:
    name: str
    V: int; D: int; R: int
    air: List[Aircraft]
    ax: np.ndarray; ay: np.ndarray; size: np.ndarray; score: np.ndarray; o: np.ndarray; c: np.ndarray
    bx: np.ndarray; by: np.ndarray
    # derived
    tt: List[np.ndarray] = field(default_factory=list)   # tt[r][i,j] minutes over all nodes (V+D)
    tau: List[np.ndarray] = field(default_factory=list)  # tau[r][i] minutes, i in 0..V+D-1 (0 at bases)

    @property
    def N(self): return self.V + self.D
    def is_base(self, i): return i >= self.V

    def build(self):
        X = np.concatenate([self.ax, self.bx]); Y = np.concatenate([self.ay, self.by])
        dist = np.sqrt((X[:, None]-X[None, :])**2 + (Y[:, None]-Y[None, :])**2)
        self.tt, self.tau = [], []
        for a in self.air:
            self.tt.append(dist * TIME_SCALE / a.v)
            t = np.zeros(self.N); t[:self.V] = self.size * SIZE_SCALE * TIME_SCALE / (a.w * a.v)
            self.tau.append(t)
        return self

def read_instance(path: str, name: Optional[str] = None) -> Instance:
    """Reads the adapted-Cordeau format (see Description.txt). Base numbers in the file are V+1..V+D."""
    with open(path) as f:
        lines = [ln.split() for ln in f if ln.strip()]
    R, V, D = map(int, lines[0][:3])
    air = []
    for k in range(R):
        T, v, w = map(float, lines[1+k][:3]); nb = int(float(lines[1+k][3]))
        start = (nb - V - 1) if nb > 0 else -1
        air.append(Aircraft(T, v, w, start))
    A = np.array([[float(x) for x in ln[:7]] for ln in lines[1+R:1+R+V]])
    B = np.array([[float(x) for x in ln[:3]] for ln in lines[1+R+V:1+R+V+D]])
    inst = Instance(name or os.path.basename(path), V, D, R, air, A[:, 1], A[:, 2], A[:, 3], A[:, 4], A[:, 5], A[:, 6], B[:, 1], B[:, 2])
    return inst.build()

def write_instance(inst: Instance, path: str):
    with open(path, 'w') as f:
        f.write(f"{inst.R} {inst.V} {inst.D}\n")
        for a in inst.air:
            f.write(f"{a.T:g} {a.v:g} {a.w:g} {inst.V+1+a.start if a.start >= 0 else 0}\n")
        for i in range(inst.V):
            f.write(f"{i+1} {inst.ax[i]:.3f} {inst.ay[i]:.3f} {inst.size[i]:g} {inst.score[i]:g} {inst.o[i]:g} {inst.c[i]:g}\n")
        for k in range(inst.D):
            f.write(f"{inst.V+1+k} {inst.bx[k]:.3f} {inst.by[k]:.3f} 0 1 0 1000\n")

# =====================================================================================
#  Route evaluation (exact minimum duration with time windows and free departure time)
# =====================================================================================
def schedule(inst: Instance, r: int, route: List[int], start_base: int, end_base: int):
    """route = list of area indices (may be empty).
    Returns (feasible, duration, d_max, cum_end) where d_max is the latest feasible departure time.
    Formulation: arrival at k-th stop = max(d + cum_k, max_{m<k}(o_m + cum_k - cum_m)); start of service >= o_k,
    service must start <= c_k; duration = end - d, minimised by d = d_max = min_k (c_k - cum_k) (>= 0 required)."""
    tt, tau = inst.tt[r], inst.tau[r]
    if not route:
        return True, 0.0, 0.0, 0.0
    T = inst.air[r].T
    nodes = [inst.V + start_base] + list(route) + [inst.V + end_base]
    cum = 0.0; H = 0.0          # H = max_{m<k} (o_m - cum_m), base has o=0,cum=0 -> H starts at 0 (d>=0)
    dmax = INF
    prev = nodes[0]
    for k in nodes[1:]:
        cum += tau[prev] + tt[prev, k]
        if k < inst.V:
            g = inst.c[k] - cum
            if g < H - 1e-9:        # some earlier window forces arrival after c_k
                return False, INF, -1.0, cum
            dmax = min(dmax, g)
            H = max(H, inst.o[k] - cum)
        prev = k
    if dmax < -1e-9:
        return False, INF, dmax, cum
    dmax = max(0.0, min(dmax, 1e12))
    dur = float(cum + max(0.0, H - dmax))      # end = max(d+cum, max_m(o_m)+cum-cum_m) with d = dmax
    # d = dmax minimises duration (duration non-increasing in d). If dmax is +inf (no windows) d=0 is fine.
    return bool(dur <= T + 1e-6), dur, float(dmax), float(cum)

def route_score(inst: Instance, route: List[int]) -> float:
    return float(inst.score[route].sum()) if route else 0.0

# =====================================================================================
#  Column (path) representation for the master problem
# =====================================================================================
@dataclass
class Column:
    r: int
    route: List[int]
    start: int
    end: int
    score: float
    dur: float
    def key(self): return (self.r, self.start, tuple(self.route), self.end)

# =====================================================================================
#  LP / MIP backends for the restricted master problem
# =====================================================================================
class SimplexRMP:
    """Dense primal simplex (Bland) with warm start for the RMP:
       max P z  s.t.  A z <= 1 (areas),  sum_r z <= 1 (aircraft), z >= 0.
    Slacks occupy indices 0..m-1 so that column indices are stable as columns are appended;
    the basis is kept between calls (adding columns keeps the old basis primal feasible).
    Used ONLY for testing without Gurobi. Also provides a tiny exact MIP by DFS over columns."""
    def __init__(self, inst: Instance):
        self.inst = inst; self.cols: List[Column] = []; self.keys = set()
        self.m = inst.V + inst.R; self.basis = list(range(self.m))
        self.Acols = []   # dense column vectors (length m) of path columns
        self.P = []
    def add_column(self, col: Column) -> bool:
        if col.key() in self.keys: return False
        self.keys.add(col.key()); self.cols.append(col)
        v = np.zeros(self.m)
        for i in col.route: v[i] = 1.0
        v[self.inst.V + col.r] = 1.0
        self.Acols.append(v); self.P.append(col.score); return True
    def solve_lp(self):
        m = self.m; n = len(self.cols)
        T = np.hstack([np.eye(m), np.array(self.Acols).T]) if n else np.eye(m)
        c = np.concatenate([np.zeros(m), np.array(self.P)]); b = np.ones(m)
        basis = self.basis; inb = np.zeros(m+n, bool); inb[basis] = True
        degen = 0
        for _ in range(50000):
            B = T[:, basis]
            Binv = np.linalg.inv(B)
            y = c[basis] @ Binv
            red = c - y @ T; red[inb] = -1.0
            cand = np.nonzero(red > 1e-9)[0]
            if len(cand) == 0: break
            enter = int(cand[0]) if degen > 50 else int(cand[np.argmax(red[cand])])   # Dantzig, Bland if stalling
            d = Binv @ T[:, enter]; xB = Binv @ b
            pos = np.nonzero(d > 1e-12)[0]
            if len(pos) == 0: raise RuntimeError('unbounded RMP (impossible)')
            ratios = xB[pos] / d[pos]; k = int(np.argmin(ratios)); leave = int(pos[k])
            degen = degen + 1 if ratios[k] < 1e-12 else 0
            inb[basis[leave]] = False; basis[leave] = enter; inb[enter] = True
        B = T[:, basis]; xB = np.linalg.solve(B, b); y = np.linalg.solve(B.T, c[basis])
        x = np.zeros(m+n); x[basis] = xB; self.basis = basis
        pi = np.maximum(y[:self.inst.V], 0.0); lam = np.maximum(y[self.inst.V:], 0.0)
        return float(c @ x), x[m:], pi, lam
    def solve_mip(self, time_limit=60.0):
        """Exact over the current pool by DFS with bound (pools are small in tests)."""
        cols = self.cols; V, R = self.inst.V, self.inst.R
        byr = [[j for j, c in enumerate(cols) if c.r == r] for r in range(R)]
        for r in range(R): byr[r].sort(key=lambda j: -cols[j].score)
        best = [0.0, [None]*R]; used = np.zeros(V, bool); t0 = time.time()
        def rec(r, val, chosen):
            if time.time() - t0 > time_limit: return
            if val + sum((cols[byr[q][0]].score if byr[q] else 0) for q in range(r, R)) <= best[0] + 1e-9: return
            if r == R:
                if val > best[0] + 1e-9: best[0] = val; best[1] = chosen[:]
                return
            for j in byr[r] + [None]:
                if j is None: rec(r+1, val, chosen + [None]); continue
                rt = cols[j].route
                if any(used[i] for i in rt): continue
                for i in rt: used[i] = True
                rec(r+1, val + cols[j].score, chosen + [j])
                for i in rt: used[i] = False
        rec(0, 0.0, [])
        self.mip_complete = (time.time() - t0 <= time_limit)
        return best[0], [cols[j] for j in best[1] if j is not None]

class GurobiRMP:
    """Restricted master problem on gurobipy: columns are added incrementally; LP duals; final MIP."""
    def __init__(self, inst: Instance, env=None, threads=0):
        import gurobipy as gp
        from gurobipy import GRB
        self.gp, self.GRB = gp, GRB
        self.inst = inst; self.cols: List[Column] = []; self.keys = set()
        self.m = gp.Model('RMP', env=env) if env is not None else gp.Model('RMP')
        self.m.Params.OutputFlag = 0
        if threads: self.m.Params.Threads = threads
        self.m.ModelSense = GRB.MAXIMIZE
        self.row_area = [self.m.addConstr(gp.LinExpr() <= 1, name=f'a{i}') for i in range(inst.V)]
        self.row_air = [self.m.addConstr(gp.LinExpr() <= 1, name=f'r{r}') for r in range(inst.R)]
        self.vars = []
        self.m.update()
    def add_column(self, col: Column) -> bool:
        if col.key() in self.keys: return False
        self.keys.add(col.key()); self.cols.append(col)
        gp = self.gp
        constrs = [self.row_area[i] for i in col.route] + [self.row_air[col.r]]
        coeffs = [1.0]*len(constrs)
        z = self.m.addVar(lb=0, ub=1, obj=float(col.score), vtype='C', name=f'z{len(self.cols)-1}', column=gp.Column(coeffs, constrs))
        self.vars.append(z); return True
    def solve_lp(self):
        for z in self.vars: z.VType = 'C'
        self.m.update(); self.m.optimize()
        x = np.array([z.X for z in self.vars])
        pi = np.array([max(0.0, c.Pi) for c in self.row_area]); lam = np.array([max(0.0, c.Pi) for c in self.row_air])
        return float(self.m.ObjVal), x, pi, lam
    def solve_mip(self, time_limit=600.0):
        for z in self.vars: z.VType = 'B'
        self.m.Params.TimeLimit = time_limit; self.m.update(); self.m.optimize()
        chosen = [c for c, z in zip(self.cols, self.vars) if z.X > 0.5]
        return float(self.m.ObjVal), chosen

# =====================================================================================
#  Pricing by exact enumeration (tiny instances only; used as test oracle)
# =====================================================================================
def enumerate_routes(inst: Instance, r: int, max_len: int = 6):
    """All feasible (start,end,route) for aircraft r by DFS; tiny instances only."""
    a = inst.air[r]; starts = [a.start] if a.start >= 0 else list(range(inst.D))
    out = []
    def dfs(route, start):
        for e in range(inst.D):
            ok, dur, _, _ = schedule(inst, r, route, start, e)
            if ok: out.append((start, e, list(route), dur))
        if len(route) >= max_len: return
        for i in range(inst.V):
            if i in route: continue
            # prune: partial route must be feasible to at least one end base
            if any(schedule(inst, r, route+[i], start, e)[0] for e in range(inst.D)):
                dfs(route+[i], start)
    for s in starts: dfs([], s)
    return out

def brute_force_optimum(inst: Instance, max_len: int = 6):
    """Exact optimum of tiny instances: one feasible route per aircraft, areas disjoint."""
    routes = [enumerate_routes(inst, r, max_len) for r in range(inst.R)]
    best = [0.0, None]
    used = np.zeros(inst.V, bool)
    def rec(r, val, chosen):
        if r == inst.R:
            if val > best[0] + 1e-9: best[0] = val; best[1] = chosen[:]
            return
        for (s, e, rt, dur) in routes[r]:
            if any(used[i] for i in rt): continue
            for i in rt: used[i] = True
            rec(r+1, val + route_score(inst, rt), chosen + [(s, e, rt)])
            for i in rt: used[i] = False
    rec(0, 0.0, [])
    return best[0], best[1]

# =====================================================================================
#  Tabu Search pricing heuristic
# =====================================================================================
class TabuPricer:
    """Maximises  f(route) = sum_{j in route} (score_j - pi_j) - lambda_r  over feasible routes of aircraft r.
    Neighbourhoods: insertion (vectorised O(1) feasibility via prefix/suffix slacks), removal, swap (replace),
    2-opt segment reversal and or-opt relocation (re-evaluated with schedule()). Tabu attributes on areas."""
    def __init__(self, inst: Instance, r: int, max_iter=None, tenure=None, stall=None, seed=0, time_limit=INF):
        self.inst, self.r = inst, r
        self.a = inst.air[r]
        self.tt, self.tau = inst.tt[r], inst.tau[r]
        n = inst.V
        self.max_iter = max_iter if max_iter is not None else 60 + 8*int(math.sqrt(n))
        self.base_iter = self.max_iter
        self.tenure = tenure if tenure is not None else max(3, int(round(math.sqrt(n)/2)))
        self.stall = stall if stall is not None else max(15, self.max_iter//5)
        self.rng = random.Random(seed); self.time_limit = time_limit
        self.starts = [self.a.start] if self.a.start >= 0 else list(range(inst.D))

    # ---- scheduling helpers --------------------------------------------------------
    def _eval(self, route, s, e):
        return schedule(self.inst, self.r, route, s, e)

    def _best_end(self, route, s):
        """Best end base = the one giving minimal duration. Since bases have no window, the optimal end base is the
        one nearest (in time) to the last area; the prefix schedule is computed once."""
        inst = self.inst
        if not route:
            return True, 0.0, s
        last = route[-1]
        e = int(np.argmin(self.tt[last, inst.V:inst.V + inst.D]))
        ok, dur, _, _ = self._eval(route, s, e)
        return (ok, dur, e) if ok else (False, INF, e)

    def _insertions(self, route, s, e, cand, gain):
        """Vectorised evaluation of inserting each candidate area (array cand) at every position.
        Returns list of (delta_f, area, pos) feasible, where pos = index in route where area is inserted."""
        inst, tt, tau = self.inst, self.tt, self.tau
        V = inst.V; T = self.a.T
        nodes = [V + s] + list(route) + [V + e]; L = len(nodes)
        # prefix quantities along the current route (no-wait cumulative arrival offsets)
        cum = np.zeros(L); g = np.full(L, INF); oc = np.full(L, -INF)   # oc_k = o_k - cum_k (areas), -inf at bases
        for k in range(1, L):
            cum[k] = cum[k-1] + tau[nodes[k-1]] + tt[nodes[k-1], nodes[k]]
            if nodes[k] < V: g[k] = inst.c[nodes[k]] - cum[k]; oc[k] = inst.o[nodes[k]] - cum[k]
        oc[0] = 0.0  # base: o=0, cum=0  (encodes d >= 0)
        # prefix max of oc (H_p = max_{m<=p}), prefix min of g, suffix min of g, suffix max of oc
        Hp = np.maximum.accumulate(oc); Gp = np.minimum.accumulate(g)
        Gs = np.minimum.accumulate(g[::-1])[::-1]; Os = np.maximum.accumulate(oc[::-1])[::-1]
        out = []
        o_c, c_c, tau_c = inst.o[cand], inst.c[cand], tau[cand]
        t_near = tt[np.ix_(cand, range(V, V + inst.D))].min(axis=1)   # time to the nearest base from each candidate
        for p in range(L-1):                       # insert between nodes[p] and nodes[p+1]
            i, j = nodes[p], nodes[p+1]
            cum_a = cum[p] + tau[i] + tt[i, cand]              # arrival offset at candidate
            # shift of the suffix; when inserting in last position the end base is re-chosen (nearest base, as _best_end does)
            t_out = t_near if p == L-2 else tt[cand, j]
            delta = tt[i, cand] + tau_c + t_out - tt[i, j]
            g_a = c_c - cum_a; oc_a = o_c - cum_a
            Hpp = Hp[p]
            feas = (g_a >= Hpp - 1e-9)                          # window of a reachable given earlier waits
            Gsuf = Gs[p+1] - delta                              # suffix slack after shift
            feas &= (Gsuf >= np.maximum(Hpp, oc_a) - 1e-9)
            dmax = np.minimum(np.minimum(Gp[p], g_a), Gsuf)
            feas &= (dmax >= -1e-9)
            Hall = np.maximum(np.maximum(Hpp, oc_a), Os[p+1] - delta)
            cum_end = cum[-1] + delta
            dur = cum_end + np.maximum(0.0, Hall - np.minimum(np.maximum(dmax, 0.0), 1e12))
            feas &= (dur <= T + 1e-6)
            for q in np.nonzero(feas)[0]:
                out.append((float(gain[q]), int(cand[q]), p))
        return out

    # ---- main search --------------------------------------------------------------
    def search(self, pi: np.ndarray, lam_r: float, init_route: Optional[List[int]] = None, init_start: Optional[int] = None,
               pool_size: int = 5, min_rc: float = 1e-4, restarts: int = 3):
        """Runs `restarts` tabu searches (warm start, empty start, perturbed best) and merges the column pools.
        Returns (best_f, best_route, (start,end), pool) where pool = list of (f, route, start, end) with f > min_rc."""
        self._gain = self.inst.score - pi; self._lam = lam_r; self._pool = {}; self._min_rc = min_rc
        self._t0 = time.time()
        best = (-INF, [], (self.starts[0], 0))
        inits = [(init_route, init_start), ([], None)]
        for k in range(restarts):
            if k < len(inits): ir, iss = inits[k]
            else:
                ir = list(best[1]); iss = best[2][0]
                for _ in range(min(2, len(ir))):
                    if ir: ir.pop(self.rng.randrange(len(ir)))
            f, rt, se = self._run(ir, iss)
            if f > best[0] + 1e-9: best = (f, rt, se)
            if time.time() - self._t0 > self.time_limit: break
        cols = sorted(self._pool.values(), key=lambda x: -x[0])[:pool_size]
        return best[0], best[1], best[2], cols

    def _consider(self, rt, ss, ee, val):
        if val > self._min_rc:
            k = (ss, tuple(rt), ee)
            if k not in self._pool: self._pool[k] = (val, list(rt), ss, ee)

    def _run(self, init_route, init_start):
        """One tabu search trajectory. Move selection per iteration:
           1. best admissible insertion if it improves f (areas with positive gain only);
           2. otherwise the intra-route move (2-opt / or-opt) that most reduces the route duration (makes room);
           3. otherwise the least-damaging removal (or a change of departure base), accepting a worsening step.
           Tabu: an inserted area cannot be removed, and a removed area cannot be re-inserted, for `tenure` iterations
           (aspiration if the move yields a new best). Diversification after `stall` non-improving iterations."""
        inst = self.inst; V = inst.V; gain_all = self._gain; lam_r = self._lam
        s = init_start if init_start is not None else self.rng.choice(self.starts)
        route = list(init_route) if init_route else []
        ok, dur, e = self._best_end(route, s)
        if not ok: route = []; ok, dur, e = self._best_end(route, s)
        f = lambda rt: float(gain_all[rt].sum()) - lam_r if rt else -lam_r
        cur_f = f(route); best_f, best_route, best_se = cur_f, list(route), (s, e)
        self._consider(route, s, e, cur_f)
        tabu_in = {}; tabu_out = {}
        it = 0; last_imp = 0
        attractive = np.nonzero(gain_all > 1e-9)[0]
        while it < self.max_iter and time.time() - self._t0 < self.time_limit:
            it += 1
            in_route = np.zeros(V, bool); in_route[route] = True
            cand = attractive[~in_route[attractive]]
            if len(cand):
                keep = [tabu_out.get(int(a), -1) < it or (cur_f + gain_all[a] > best_f + 1e-9) for a in cand]
                cand = cand[keep]
            applied = False
            # 1) insertions
            if len(cand):
                ins = self._insertions(route, s, e, cand, gain_all[cand])
                if ins:
                    self.rng.shuffle(ins); gn, a, p = max(ins, key=lambda m: m[0])
                    if gn > 1e-9:
                        route = route[:p] + [a] + route[p:]; ok, dur, e = self._best_end(route, s)
                        tabu_in[a] = it + self.tenure; applied = True
            # 2) intra-route moves that reduce duration (make room for further insertions)
            if not applied and len(route) >= 2:
                L = len(route); best_mv = (dur - 1e-6, None)
                for k1 in range(L-1):
                    for k2 in range(k1+1, L):
                        rt = route[:k1] + route[k1:k2+1][::-1] + route[k2+1:]
                        okk, d2, ee = self._best_end(rt, s)
                        if okk and d2 < best_mv[0]: best_mv = (d2, (rt, ee))
                for k in range(L):
                    a = route[k]; rest = route[:k] + route[k+1:]
                    for p in range(len(rest)+1):
                        if p == k: continue
                        rt = rest[:p] + [a] + rest[p:]
                        okk, d2, ee = self._best_end(rt, s)
                        if okk and d2 < best_mv[0]: best_mv = (d2, (rt, ee))
                if best_mv[1] is not None:
                    route, e = best_mv[1]; dur = best_mv[0]; applied = True
            # 3) worsening step: removal (least loss) or base change
            if not applied:
                moves = []
                for k in range(len(route)):
                    a = route[k]
                    if tabu_in.get(a, -1) >= it and not (cur_f - gain_all[a] > best_f + 1e-9): continue
                    moves.append((cur_f - gain_all[a], 'rem', k))
                if len(self.starts) > 1:
                    for ss in self.starts:
                        if ss != s:
                            okk, d2, ee = self._best_end(route, ss)
                            if okk: moves.append((cur_f - 1e-6, 'base', (ss, ee)))
                if not moves: break
                self.rng.shuffle(moves); mv = max(moves, key=lambda m: m[0])
                if mv[1] == 'rem':
                    k = mv[2]; a = route[k]; route = route[:k] + route[k+1:]; tabu_out[a] = it + self.tenure
                    ok, dur, e = self._best_end(route, s)
                else:
                    s, e = mv[2]; ok, dur, _ = self._best_end(route, s)
            cur_f = f(route)
            self._consider(route, s, e, cur_f)
            if cur_f > best_f + 1e-9:
                best_f, best_route, best_se = cur_f, list(route), (s, e); last_imp = it
            elif it - last_imp > self.stall:
                route = list(best_route); s, e = best_se
                for _ in range(min(2, len(route))):
                    if route: route.pop(self.rng.randrange(len(route)))
                ok, dur, e = self._best_end(route, s); cur_f = f(route); last_imp = it
                tabu_in.clear(); tabu_out.clear()
        return best_f, best_route, best_se

# =====================================================================================
#  Exact pricing with Gurobi (per aircraft)
# =====================================================================================
class GurobiPricer:
    """Elementary shortest path with resource (time windows + endurance), MTZ-style, one aircraft.
    Objective: max sum (score_j - pi_j) x_ij - lambda_r.  Solution pool harvested for extra columns."""
    def __init__(self, inst: Instance, r: int, env=None, threads=0, time_limit=60.0, pool=10):
        import gurobipy as gp
        from gurobipy import GRB
        self.gp, self.GRB, self.inst, self.r = gp, GRB, inst, r
        a = inst.air[r]; V, D = inst.V, inst.D
        tt, tau = inst.tt[r], inst.tau[r]
        m = gp.Model(f'SP{r}', env=env) if env is not None else gp.Model(f'SP{r}')
        m.Params.OutputFlag = 0; m.Params.TimeLimit = time_limit
        m.Params.PoolSolutions = pool; m.Params.PoolSearchMode = 0
        if threads: m.Params.Threads = threads
        self.time_limit = time_limit
        starts = [a.start] if a.start >= 0 else list(range(D))
        self.starts = starts
        S = [V + s for s in starts]; E = [V + e for e in range(D)]
        A = list(range(V))
        # arcs: S->A, A->A, A->E
        arcs = [(i, j) for i in S for j in A] + [(i, j) for i in A for j in A if i != j] + [(i, j) for i in A for j in E]
        self.x = m.addVars(arcs, vtype=GRB.BINARY, name='x')
        # time variables: s[i] = start of service at area i; s[V+d] = DEPARTURE time from base d; e[d] = ARRIVAL time at
        # base d (separate variable, so that a route may return to its departure base). A departure later than the last
        # window closing is useless, hence the upper bound c_max, which also keeps every big-M small.
        c_max = float(inst.c.max())
        self.s = m.addVars(range(V + D), lb=0.0, ub=[GRB.INFINITY] * V + [c_max] * D, name='s')
        self.e = m.addVars(range(D), lb=0.0, name='e')
        # one departure, one arrival, flow conservation
        m.addConstr(gp.quicksum(self.x[i, j] for i in S for j in A) == 1)
        m.addConstr(gp.quicksum(self.x[i, j] for i in A for j in E) == 1)
        for h in A:
            m.addConstr(gp.quicksum(self.x[i, h] for i in S + A if i != h) == gp.quicksum(self.x[h, j] for j in A + E if j != h))
            m.addConstr(gp.quicksum(self.x[i, h] for i in S + A if i != h) <= 1)          # elementary: visit h at most once
        # endurance as a knapsack on travel + service time (valid: waiting >= 0); tightens the LP relaxation (paper eq. 7)
        m.addConstr(gp.quicksum(float(tt[i, j] + tau[i]) * self.x[i, j] for (i, j) in arcs) <= float(a.T))
        # time propagation with tight big-M; service time at bases is 0
        o = np.concatenate([inst.o, np.zeros(D)]); c = np.concatenate([inst.c, np.full(D, c_max)])
        for (i, j) in arcs:
            M = float(max(0.0, c[i] + tau[i] + tt[i, j] - o[j]) + 1.0)
            tj = self.e[j - V] if j >= V else self.s[j]
            m.addConstr(self.s[i] + float(tau[i] + tt[i, j]) - tj <= M * (1 - self.x[i, j]))
        for i in A:
            m.addConstr(self.s[i] >= float(inst.o[i])); m.addConstr(self.s[i] <= float(inst.c[i]))
        for e in E:
            for st in S:
                m.addConstr(self.e[e - V] <= float(a.T) + self.s[st])
        m.ModelSense = GRB.MAXIMIZE
        self.m = m; self.A, self.S, self.E = A, S, E

    def solve(self, pi: np.ndarray, lam_r: float, min_rc: float = 1e-4, pool_size: int = 10):
        gp, GRB, inst = self.gp, self.GRB, self.inst
        gain = inst.score - pi
        obj = gp.quicksum(float(gain[j]) * self.x[i, j] for (i, j) in self.x.keys() if j < inst.V) - float(lam_r)
        self.m.setObjective(obj, GRB.MAXIMIZE)
        self.m.optimize()
        st = self.m.Status
        if st in (GRB.INFEASIBLE, GRB.INF_OR_UNBD):
            return True, 0.0, []                     # no feasible route for this aircraft: nothing to price
        try: bound = float(self.m.ObjBound)
        except Exception: bound = INF
        if bound != bound or bound > 1e100: bound = INF     # NaN / +inf guard (no bound available yet)
        proved = (st == GRB.OPTIMAL)
        cols = []
        if self.m.SolCount:
            for k in range(min(self.m.SolCount, pool_size)):
                self.m.Params.SolutionNumber = k
                val = self.m.PoolObjVal
                if val <= min_rc: continue
                xv = {key: v.Xn for key, v in self.x.items() if v.Xn > 0.5}
                route, s, e = self._extract(xv)
                if route is not None: cols.append((val, route, s, e))
        return proved, float(bound), cols

    def _extract(self, xv):
        inst = self.inst; V = inst.V
        succ = {i: j for (i, j) in xv}
        for st in self.S:
            if st in succ:
                route = []; cur = succ[st]
                while cur < V:
                    route.append(cur); cur = succ.get(cur, None)
                    if cur is None: return None, None, None
                return route, st - V, cur - V
        return None, None, None

# =====================================================================================
#  Column generation driver
# =====================================================================================
@dataclass
class CGResult:
    method: str; lp_bound: float; lagr_bound: float; integer_value: float; gap: float
    time_total: float; time_lp: float; iterations: int; n_columns: int; proved: bool
    routes: list; log: list
    hit_time_limit: bool = False     # True when the CG loop was cut by the time limit (not by convergence)

def initial_columns(inst: Instance, rmp):
    """One column per (aircraft, area) if feasible, choosing the best end base (Java: single-area routes)."""
    n = 0
    for r in range(inst.R):
        a = inst.air[r]; starts = [a.start] if a.start >= 0 else list(range(inst.D))
        for s in starts:
            for i in range(inst.V):
                best = None
                for e in range(inst.D):
                    ok, dur, _, _ = schedule(inst, r, [i], s, e)
                    if ok and (best is None or dur < best[0]): best = (dur, e)
                if best is not None:
                    n += rmp.add_column(Column(r, [i], s, best[1], float(inst.score[i]), best[0]))
    return n

def column_generation(inst: Instance, rmp, pricers_ts: Optional[list], pricers_exact: Optional[list],
                      time_limit=3600.0, max_iter=10000, min_rc=1e-4, verbose=True, final_mip_time=600.0,
                      ts_init_from_best=True, post_improve=True, sp_time_final=600.0):
    """Generic CG loop.  Each iteration: solve RMP LP, then for each aircraft price with TS (if given);
    aircraft for which TS finds no improving column are priced exactly (if exact pricers given).
    Terminates when no improving column exists (proved if all exact pricers ended OPTIMAL) or on time limit.
    Returns CGResult with LP bound, Lagrangian bound (valid even on early stop) and the integer value of the RMP MIP."""
    t0 = time.time(); log = []; t_lp = 0.0
    n0 = initial_columns(inst, rmp)
    if verbose: print(f'  initial columns: {n0}')
    it = 0; lp_val = 0.0; lagr = INF; proved = False; best_routes = {}; second_chance = False; n_second = 0
    converged = False; cut_by_time = False
    pi = np.zeros(inst.V); lam = np.zeros(inst.R)
    while it < max_iter and time.time() - t0 < time_limit:
        it += 1
        t1 = time.time(); lp_val, x, pi, lam = rmp.solve_lp(); t_lp += time.time() - t1
        # remember the best current route per aircraft to warm-start TS
        if ts_init_from_best:
            for j, c in enumerate(rmp.cols):
                if x[j] > 0.5: best_routes[c.r] = (c.route, c.start)
        new = 0; rc_report = []
        # ---- pass 1: Tabu Search pricing for every aircraft; add every new column it finds ----
        if pricers_ts is not None:
            for r in range(inst.R):
                ir, iss = best_routes.get(r, (None, None))
                bf, brt, bse, cols = pricers_ts[r].search(pi, lam[r], init_route=ir, init_start=iss, min_rc=min_rc,
                                                          restarts=(6 if second_chance else 3), pool_size=8)
                rc_report.append(bf)
                for (val, route, s, e) in cols:
                    ok, dur, _, _ = schedule(inst, r, route, s, e)
                    assert ok, 'TS returned an infeasible route'
                    new += rmp.add_column(Column(r, list(route), s, e, route_score(inst, route), dur))
        # ---- pass 2: exact pricing whenever TS added no NEW column (certification or rescue). Its best bound gives a
        #      valid Lagrangian upper bound even when it finds new columns. ----
        all_proved = False; slack = INF
        if new == 0 and pricers_exact is not None:
            all_proved = True; slack = 0.0; rc_report = []
            remaining = time_limit - (time.time() - t0)
            per_sp = max(1.0, min(sp_time_final, remaining / max(1, inst.R)))
            for r in range(inst.R):
                remaining = time_limit - (time.time() - t0)
                if remaining <= 0:
                    all_proved = False; slack = INF; cut_by_time = True; break
                pricers_exact[r].m.Params.TimeLimit = max(1.0, min(per_sp, remaining))
                pr, bound, cols = pricers_exact[r].solve(pi, lam[r], min_rc=min_rc)
                rc_report.append(bound)
                all_proved = all_proved and pr
                slack = slack + max(0.0, bound) if (slack < INF and bound < INF) else INF
                for (val, route, s, e) in cols:
                    ok, dur, _, _ = schedule(inst, r, route, s, e)
                    assert ok, 'exact pricer returned an infeasible route'
                    new += rmp.add_column(Column(r, list(route), s, e, route_score(inst, route), dur))
        lagr_it = lp_val + slack if (pricers_exact is not None and slack < INF) else INF
        if lagr_it < lagr: lagr = lagr_it
        log.append(dict(it=it, t=round(time.time()-t0, 1), lp=round(lp_val, 3), lagr=(None if lagr == INF else round(lagr, 3)),
                        ncols=len(rmp.cols), new=new, max_rc=round(max(rc_report) if rc_report else 0, 4)))
        if verbose and (it % 5 == 1 or new == 0):
            print(f"  it {it:4d}  t={time.time()-t0:7.1f}s  LP={lp_val:9.3f}  lagr={'-' if lagr==INF else f'{lagr:9.3f}'}  cols={len(rmp.cols):5d}  new={new}")
        if new == 0:
            if pricers_exact is None and n_second < 2:
                # heuristic pricing found nothing: intensify (more restarts/iterations) before giving up
                second_chance = True; n_second += 1
                for p in pricers_ts: p.max_iter = p.base_iter * 3
                continue
            proved = all_proved if pricers_exact is not None else False
            converged = not cut_by_time
            break
        if second_chance:
            second_chance = False
            for p in pricers_ts: p.max_iter = p.base_iter
    if pricers_ts is not None:
        for p in pricers_ts: p.max_iter = p.base_iter
    if pricers_exact is not None and proved:
        lagr = min(lagr, lp_val)      # when pricing is proved, the RMP LP value is the LP bound itself
    hit_time_limit = not converged
    t1 = time.time(); int_val, chosen = rmp.solve_mip(time_limit=final_mip_time); t_mip = time.time() - t1
    # post-optimisation: sequential re-pricing of each route given the others (TS with used areas forbidden)
    if pricers_ts is not None and post_improve:
        chosen, int_val = improve_integer(inst, chosen, pricers_ts)
    ub = lagr if lagr < INF else float('nan')
    gap = (ub - int_val) / ub * 100 if ub == ub and ub > 0 else float('nan')
    method = 'CG' if pricers_exact is not None else 'CG-TS'
    routes = [dict(r=c.r, start=c.start, end=c.end, route=[int(i) for i in c.route], score=float(c.score), dur=round(float(c.dur), 2)) for c in chosen]
    return CGResult(method, lp_val, ub, int_val, gap, time.time()-t0, t_lp, it, len(rmp.cols), proved, routes, log, hit_time_limit)

def improve_integer(inst: Instance, chosen: List[Column], pricers_ts: list, rounds: int = 3):
    """Given one route per aircraft (list of Column), re-optimise each route with TS while the other routes are fixed
    (areas used by others get a prohibitive dual). Accepts only improvements; returns (columns, total score)."""
    cur = {c.r: c for c in chosen}
    for _ in range(rounds):
        improved = False
        for r in range(inst.R):
            used = set()
            for q, c in cur.items():
                if q != r: used.update(c.route)
            pi = np.zeros(inst.V); pi[list(used)] = 1e9
            c0 = cur.get(r)
            f, rt, (s, e), _ = pricers_ts[r].search(pi, 0.0, init_route=(c0.route if c0 else None), init_start=(c0.start if c0 else None),
                                                   restarts=4, pool_size=1, min_rc=INF)
            if rt and (c0 is None or f > c0.score + 1e-9):
                ok, dur, _, _ = schedule(inst, r, rt, s, e); assert ok
                cur[r] = Column(r, list(rt), s, e, route_score(inst, rt), dur); improved = True
        if not improved: break
    cols = [cur[r] for r in sorted(cur)]
    return cols, float(sum(c.score for c in cols))

# =====================================================================================
#  Compact MILP with Gurobi
# =====================================================================================
def solve_milp(inst: Instance, time_limit=3600.0, env=None, threads=0, verbose=False, mip_gap=1e-4):
    """Compact MILP (same as the paper's model, with tight big-M per arc). Returns dict with value, bound, gap, time, routes."""
    import gurobipy as gp
    from gurobipy import GRB
    V, D, R = inst.V, inst.D, inst.R
    m = gp.Model('MBH-TOPTW', env=env) if env is not None else gp.Model('MBH-TOPTW')
    m.Params.OutputFlag = 1 if verbose else 0; m.Params.TimeLimit = time_limit; m.Params.MIPGap = mip_gap
    if threads: m.Params.Threads = threads
    A = list(range(V)); E = [V + e for e in range(D)]
    c_max = float(inst.c.max())        # no useful departure after the last window closes; also keeps the big-M small
    o = np.concatenate([inst.o, np.zeros(D)]); c = np.concatenate([inst.c, np.full(D, c_max)])
    x = {}; s = {}; inc = {h: [] for h in A}
    t_build = time.time()
    for r in range(R):
        a = inst.air[r]; S = [V + a.start] if a.start >= 0 else [V + d for d in range(D)]
        tt, tau = inst.tt[r], inst.tau[r]
        arcs = [(i, j) for i in S for j in A] + [(i, j) for i in A for j in A if i != j] + [(i, j) for i in A for j in E]
        xr = m.addVars(arcs, vtype=GRB.BINARY, name=f'x{r}')
        # sr[i]: service start at area i; sr[V+d]: departure time from base d; er[d]: arrival time at base d (separate
        # variable, so that an aircraft may return to its departure base)
        sr = m.addVars(range(V + D), lb=0.0, ub=[GRB.INFINITY] * V + [c_max] * D, name=f's{r}')
        er = m.addVars(range(D), lb=0.0, name=f'e{r}')
        x[r], s[r] = xr, sr
        m.addConstr(gp.quicksum(xr[i, j] for i in S for j in A) <= 1)
        m.addConstr(gp.quicksum(xr[i, j] for i in A for j in E) <= 1)
        for h in A:
            ins = [xr[i, h] for i in S + A if i != h]; inc[h].extend(ins)
            m.addConstr(gp.quicksum(ins) == gp.quicksum(xr[h, j] for j in A + E if j != h))
        m.addConstr(gp.quicksum(float(tt[i, j] + tau[i]) * xr[i, j] for (i, j) in arcs) <= float(a.T))   # paper eq. (7)
        for (i, j) in arcs:
            M = float(max(0.0, c[i] + tau[i] + tt[i, j] - o[j]) + 1.0)
            tj = er[j - V] if j >= V else sr[j]
            m.addConstr(sr[i] + float(tau[i] + tt[i, j]) - tj <= M * (1 - xr[i, j]))
        for i in A:
            m.addConstr(sr[i] >= float(inst.o[i])); m.addConstr(sr[i] <= float(inst.c[i]))
        for e in E:
            for st in S:
                m.addConstr(er[e - V] <= float(a.T) + sr[st])
    for h in A:
        m.addConstr(gp.quicksum(inc[h]) <= 1)
    m.setObjective(gp.quicksum(float(inst.score[j]) * x[r][i, j] for r in range(R) for (i, j) in x[r].keys() if j < V), GRB.MAXIMIZE)
    t_build = time.time() - t_build
    m.optimize()
    val = m.ObjVal if m.SolCount else 0.0; bound = m.ObjBound
    gap = (bound - val) / bound * 100 if bound > 0 else 0.0
    routes = []
    if m.SolCount:
        for r in range(R):
            succ = {i: j for (i, j), v in x[r].items() if v.X > 0.5}
            S = [V + inst.air[r].start] if inst.air[r].start >= 0 else [V + d for d in range(D)]
            for st in S:
                if st in succ:
                    rt = []; cur = succ[st]
                    while cur < V: rt.append(cur); cur = succ[cur]
                    ok, dur, _, _ = schedule(inst, r, rt, st - V, cur - V)
                    routes.append(dict(r=int(r), start=int(st-V), end=int(cur-V), route=[int(i) for i in rt], score=float(route_score(inst, rt)),
                                       dur=round(float(dur), 2), feasible=bool(ok)))
    status = {GRB.OPTIMAL: 'optimal', GRB.TIME_LIMIT: 'time_limit'}.get(m.Status, str(m.Status))
    out = dict(method='MILP', value=float(val), bound=float(bound), gap=float(gap), time=float(m.Runtime), time_build=t_build,
               status=status, nodes=int(m.NodeCount), vars=m.NumVars, constrs=m.NumConstrs, routes=routes)
    m.dispose()
    return out

# =====================================================================================
#  Convenience runners
# =====================================================================================
def run_cg(inst: Instance, mode='CG', time_limit=3600.0, env=None, threads=0, sp_time=60.0, verbose=True,
           ts_iter=None, seed=0, final_mip_time=600.0, sp_time_final=600.0):
    """mode='CG' (TS + exact certification) or 'CG-TS' (TS only). Without gurobipy falls back to the test simplex
    backend and TS-only pricing (for local testing)."""
    try:
        import gurobipy  # noqa
        have_grb = True
    except Exception:
        have_grb = False
    ts = [TabuPricer(inst, r, max_iter=ts_iter, seed=seed + r) for r in range(inst.R)]
    if have_grb:
        rmp = GurobiRMP(inst, env=env, threads=threads)
        ex = [GurobiPricer(inst, r, env=env, threads=threads, time_limit=sp_time) for r in range(inst.R)] if mode == 'CG' else None
    else:
        rmp = SimplexRMP(inst); ex = None
        if mode == 'CG': print('  [warning] gurobipy not available: CG runs as CG-TS on the test backend')
    res = column_generation(inst, rmp, ts, ex, time_limit=time_limit, verbose=verbose, final_mip_time=final_mip_time,
                            sp_time_final=max(sp_time, sp_time_final))
    if have_grb:
        for p in (ex or []): p.m.dispose()
        rmp.m.dispose()
    return res

def verify_solution(inst: Instance, routes: list) -> Tuple[bool, float]:
    """Independent check: feasibility of every route and disjointness of areas. Returns (ok, total_score)."""
    used = set(); total = 0.0; ok = True
    for rt in routes:
        f, dur, _, _ = schedule(inst, rt['r'], rt['route'], rt['start'], rt['end'])
        ok &= f
        for i in rt['route']:
            if i in used: ok = False
            used.add(i)
        total += route_score(inst, rt['route'])
    return ok, total

In [ ]:
%%writefile /content/mbh_v2/code/mbh_instances.py
"""
mbh_instances.py -- instance generation for the MBH-TOPTW computational study.

1. Heterogeneous adapted-Cordeau benchmark (pr01..pr20):
   the original adaptation uses one aircraft type per instance (T, v, w). We assign aircraft k (0-based) the profile
   PROFILES[k mod 3] = multipliers on (T, v, w):
       A: (1.00, 1.00, 1.0)   baseline
       B: (0.80, 1.25, 0.6)   fast, short-legged, narrow sensor
       C: (1.25, 0.80, 1.6)   slow, long-legged, wide sensor
   Travel times and service times therefore differ by aircraft (service time ~ 1/(w*v): B is 1.33x slower, C 0.78x).
2. Scalability families: pr02 truncated to its first n areas, n in {20,...,90} (3 aircraft), and pr06 truncated to
   n in {40,...,280} (7 aircraft), the family on which the compact MILP stops closing.
3. Random instances (Table 4 bounds), heterogeneous by construction, fixed seeds.
4. Real instance of Section 6.4 (8 Chilean MPAs, 4 bases, P-3 and C-295) from the authors' data sheet: build_real_instance.
"""
from __future__ import annotations
import os, math, json
import numpy as np
from mbh_core import Instance, Aircraft, read_instance, write_instance

PROFILES = [(1.00, 1.00, 1.0), (0.80, 1.25, 0.6), (1.25, 0.80, 1.6)]
PROFILE_NAMES = ['A', 'B', 'C']

def make_heterogeneous(inst: Instance) -> Instance:
    air = []
    for k, a in enumerate(inst.air):
        fT, fv, fw = PROFILES[k % 3]
        air.append(Aircraft(float(round(a.T * fT)), float(round(a.v * fv)), float(round(a.w * fw, 1)), a.start))
    out = Instance(inst.name, inst.V, inst.D, inst.R, air, inst.ax, inst.ay, inst.size, inst.score, inst.o, inst.c, inst.bx, inst.by)
    return out.build()

def truncate_areas(inst: Instance, n: int, name: str) -> Instance:
    out = Instance(name, n, inst.D, inst.R, [Aircraft(a.T, a.v, a.w, a.start) for a in inst.air],
                   inst.ax[:n], inst.ay[:n], inst.size[:n], inst.score[:n], inst.o[:n], inst.c[:n], inst.bx, inst.by)
    return out.build()

# Table-4 bounds
RB = dict(T=(400, 800), v=(170, 500), w=(1.0, 3.0), xy=(-100, 100), size=(1, 25), score=(1, 25), o=(0, 500), width=(90, 180))

def random_instance(R: int, V: int, D: int, seed: int, name: str) -> Instance:
    rng = np.random.default_rng(seed)
    air = [Aircraft(float(rng.integers(RB['T'][0], RB['T'][1] + 1)), float(rng.integers(RB['v'][0], RB['v'][1] + 1)),
                    float(round(rng.uniform(*RB['w']), 1)), int(rng.integers(0, D))) for _ in range(R)]
    ax = rng.uniform(*RB['xy'], V).round(3); ay = rng.uniform(*RB['xy'], V).round(3)
    size = rng.integers(RB['size'][0], RB['size'][1] + 1, V).astype(float)
    score = rng.integers(RB['score'][0], RB['score'][1] + 1, V).astype(float)
    o = rng.integers(RB['o'][0], RB['o'][1] + 1, V).astype(float)
    c = o + rng.integers(RB['width'][0], RB['width'][1] + 1, V)
    bx = rng.uniform(*RB['xy'], D).round(3); by = rng.uniform(*RB['xy'], D).round(3)
    return Instance(name, V, D, R, air, ax, ay, size, score, o, c.astype(float), bx, by).build()

# scenarios of Table 1 / Table 5 (aircraft, areas, bases)
SCENARIOS = [(2, 48, 4), (3, 96, 4), (4, 144, 4), (5, 192, 4), (6, 240, 4), (7, 288, 4), (2, 72, 6), (3, 144, 6), (4, 216, 6), (5, 288, 6),
             (1, 48, 4), (2, 96, 4), (3, 144, 4), (4, 192, 4), (5, 240, 4), (6, 288, 4), (1, 72, 6), (2, 144, 6), (3, 216, 6), (4, 288, 6)]

SCAL_HARD_SIZES = (40, 80, 120, 160, 200, 240, 280)

def generate_all(src_dir: str, out_dir: str, n_rep: int = 5, scal_sizes=(20, 30, 40, 50, 60, 70, 80, 90), scal_hard_sizes=SCAL_HARD_SIZES):
    """Writes benchmark_het/, scalability/, random/ under out_dir. Returns a manifest dict."""
    man = {'benchmark_het': [], 'scalability': [], 'scalability_hard': [], 'random': []}
    d1 = os.path.join(out_dir, 'benchmark_het'); os.makedirs(d1, exist_ok=True)
    for k in range(1, 21):
        nm = f'pr{k:02d}'; inst = make_heterogeneous(read_instance(os.path.join(src_dir, nm), nm))
        write_instance(inst, os.path.join(d1, nm)); man['benchmark_het'].append(nm)
    d2 = os.path.join(out_dir, 'scalability'); os.makedirs(d2, exist_ok=True)
    base = make_heterogeneous(read_instance(os.path.join(src_dir, 'pr02'), 'pr02'))
    for n in scal_sizes:
        nm = f'pr02_n{n:03d}'; write_instance(truncate_areas(base, n, nm), os.path.join(d2, nm)); man['scalability'].append(nm)
    d2h = os.path.join(out_dir, 'scalability_hard'); os.makedirs(d2h, exist_ok=True)
    base6 = make_heterogeneous(read_instance(os.path.join(src_dir, 'pr06'), 'pr06'))
    for n in scal_hard_sizes:
        nm = f'pr06_n{n:03d}'; write_instance(truncate_areas(base6, n, nm), os.path.join(d2h, nm)); man['scalability_hard'].append(nm)
    d3 = os.path.join(out_dir, 'random'); os.makedirs(d3, exist_ok=True)
    for si, (R, V, D) in enumerate(SCENARIOS, start=1):
        for rep in range(1, n_rep + 1):
            nm = f'rnd{si:02d}_{R}a{V}v{D}b_r{rep}'; seed = 1000 * si + rep
            write_instance(random_instance(R, V, D, seed, nm), os.path.join(d3, nm)); man['random'].append(nm)
    with open(os.path.join(out_dir, 'manifest.json'), 'w') as f: json.dump(man, f, indent=1)
    return man

# -------------------------------------------------------------------------------------------------
#  Real instance (Section 7): a single operational day over the Chilean MPAs of the authors' data sheet (version 2).
#
#  * Areas: the 8 MPAs of the sheet. The three oceanic parks and the austral park are divided into patrol SECTORS
#    (operational practice for areas of 10^5-10^6 km^2); each sector is a node with its own score and window.
#  * Fleet: three aircraft classes parameterised on public specifications (P-3ACH Orion, C-295 Persuader,
#    AS365 Dauphin class rotary-wing). Maximum route time = min(endurance, 8 h crew-duty limit).
#  * Surveillance time: Koopman random search with lateral detection range rho (paper eq. 22):
#        tau = -A ln(1-Pmin) / (2 rho v).   File stores size_eff = A(-ln(1-Pmin))/200 with w = rho (see README).
#  * Pmin per node: 0.8 for sectors up to PMIN_AREA_KM2, 0.6 for larger ones (a single pass cannot reasonably
#    achieve 0.8 over 10^5 km^2). Score s_i = round(25 * (I_i/5) * Pmin_i), I_i = strategic importance 1..5.
#  * Time windows: start of the surveillance pass between civil sunrise and sunset minus the shortest pass,
#    computed from latitude/longitude for the planning date (default 21 June, shortest day), common clock UTC-4.
#  * Bases: the four of the sheet (Iquique, Concon, Maquehue/Temuco, Punta Arenas) plus Mataveri as a forward
#    operating location for the Easter Island sectors. Each aircraft has a given departure base; arrival base free.
# -------------------------------------------------------------------------------------------------
PMIN_AREA_KM2 = 150000.0
# MPA areas from the creation decrees, rounded to the nearest thousand km2 (version 2 of the case study):
#   Rapa Nui AMCP-MU 579,368 + Motu Motiro Hiva marine park 150,000 -> 729,000
#   Nazca-Desventuradas marine park 300,035 -> 300,000
#   Juan Fernandez marine park 262,000 + AMCP-MU 24,000 (DS 12/2018) -> 286,000
#   Diego Ramirez-Paso Drake marine park 144,390.6 (DS 9/2018) -> 144,000
# Version 1 used 712,694 / 300,000 / 450,000 / 101,000 km2 and placed the Paso Drake reference point near Cape Horn,
# outside the park. Reference points of multi-sector parks: the islands for the island sectors; for Paso Drake, the
# centroid of the southern half of the park (equal-area split at 57.79 S of the official WDPA polygon).
AREA_RAPA_NUI, AREA_NAZCA, AREA_JUAN_FERNANDEZ, AREA_DIEGO_RAMIREZ_DRAKE = 729000.0, 300000.0, 286000.0, 144000.0
# (sector name, parent MPA, area km2, lat, lon, importance 1..5)
REAL_SECTORS = [
    ('Rapa Nui NE',       'Rapa Nui-Motu Motiro Hiva',  AREA_RAPA_NUI/4, -25.6, -107.8, 5),
    ('Rapa Nui NW',       'Rapa Nui-Motu Motiro Hiva',  AREA_RAPA_NUI/4, -25.6, -110.9, 5),
    ('Rapa Nui SE',       'Rapa Nui-Motu Motiro Hiva',  AREA_RAPA_NUI/4, -28.6, -107.8, 5),
    ('Rapa Nui SW',       'Rapa Nui-Motu Motiro Hiva',  AREA_RAPA_NUI/4, -28.6, -110.9, 5),
    ('Desventuradas E',   'Nazca-Desventuradas',        AREA_NAZCA/2,    -26.3,  -79.9, 5),
    ('Nazca ridge W',     'Nazca-Desventuradas',        AREA_NAZCA/2,    -26.8,  -82.0, 5),
    ('Robinson Crusoe',   'Juan Fernandez',             AREA_JUAN_FERNANDEZ/2, -33.6, -78.8, 5),
    ('Alejandro Selkirk', 'Juan Fernandez',             AREA_JUAN_FERNANDEZ/2, -33.8, -80.8, 5),
    ('Francisco Coloane', 'Francisco Coloane',               15.0, -53.65, -72.30, 4),
    ('Diego Ramirez',     'Diego Ramirez-Paso Drake',   AREA_DIEGO_RAMIREZ_DRAKE/2, -56.5,  -68.7, 5),
    ('Paso Drake',        'Diego Ramirez-Paso Drake',   AREA_DIEGO_RAMIREZ_DRAKE/2, -58.69, -70.30, 4),
    ('Isla Chanaral',     'Isla Chanaral',                   28.9, -29.03, -71.57, 3),
    ('Isla Choros',       'Isla Choros',                     38.6, -29.26, -71.54, 3),
    ('La Rinconada',      'La Rinconada',                     3.3, -23.47, -70.50, 3),
]
def _dms(d, m, s): return -(d + m / 60 + s / 3600)
REAL_BASES = [('Iquique',            (_dms(20, 32, 7),  _dms(70, 10, 53))),
              ('Concon (Torquemada)', (_dms(32, 56, 59), _dms(71, 28, 43))),
              ('Maquehue (Temuco)',   (_dms(38, 46, 1),  _dms(72, 38, 14))),
              ('Punta Arenas',        (_dms(53, 0, 9),   _dms(70, 51, 16))),
              ('Mataveri (Rapa Nui)', (-27.165, -109.422))]
CREW_LIMIT_MIN = 480.0
# (class, endurance min, speed km/h, lateral range km)  public specifications; the route limit applies min(endurance, crew limit)
REAL_CLASSES = {'P-3ACH Orion':   dict(T=720.0, v=463.0, rho=150.0),
                'C-295 Persuader': dict(T=480.0, v=480.0, rho=100.0),
                'AS365 Dauphin':   dict(T=240.0, v=250.0, rho=40.0)}
# (class, departure base) for the planning day
REAL_FLEET = [('P-3ACH Orion', 'Concon (Torquemada)'), ('P-3ACH Orion', 'Mataveri (Rapa Nui)'),
              ('C-295 Persuader', 'Iquique'), ('C-295 Persuader', 'Punta Arenas'), ('AS365 Dauphin', 'Punta Arenas')]

def sun_times_utc(lat, lon, doy):
    """NOAA simplified sunrise/sunset in minutes UTC for day-of-year doy (zenith 90.833 deg)."""
    g = 2 * math.pi / 365 * (doy - 1)
    eqt = 229.18 * (0.000075 + 0.001868 * math.cos(g) - 0.032077 * math.sin(g) - 0.014615 * math.cos(2 * g) - 0.040849 * math.sin(2 * g))
    decl = (0.006918 - 0.399912 * math.cos(g) + 0.070257 * math.sin(g) - 0.006758 * math.cos(2 * g) + 0.000907 * math.sin(2 * g)
            - 0.002697 * math.cos(3 * g) + 0.00148 * math.sin(3 * g))
    la = math.radians(lat); z = math.radians(90.833)
    cosha = math.cos(z) / (math.cos(la) * math.cos(decl)) - math.tan(la) * math.tan(decl)
    ha = math.degrees(math.acos(max(-1.0, min(1.0, cosha))))
    return 720 - 4 * (lon + ha) - eqt, 720 - 4 * (lon - ha) - eqt

def _aeqd(lat, lon, lat0, lon0, R=6371.0088):
    """Azimuthal equidistant projection (km); distance error below 2% for this instance."""
    lat, lon, lat0, lon0 = map(np.radians, (np.asarray(lat, float), np.asarray(lon, float), lat0, lon0))
    cc = np.clip(np.sin(lat0) * np.sin(lat) + np.cos(lat0) * np.cos(lat) * np.cos(lon - lon0), -1, 1); c = np.arccos(cc)
    k = np.where(c > 1e-12, c / np.sin(np.maximum(c, 1e-12)), 1.0)
    x = R * k * np.cos(lat) * np.sin(lon - lon0); y = R * k * (np.cos(lat0) * np.sin(lat) - np.sin(lat0) * np.cos(lat) * np.cos(lon - lon0))
    return x, y

def build_real_instance(doy=172, utc_offset_h=-4.0, pmin_small=0.8, pmin_large=0.6, importance=None, fleet=None, name='real_chile'):
    """Builds the Section-6.4 instance. doy: planning day (172 = 21 June). importance: optional dict sector->1..5 overrides.
    fleet: optional list of (class, base) overrides. Returns Instance with .meta describing every input."""
    secs = [list(t) for t in REAL_SECTORS]
    if importance:
        for t in secs:
            if t[0] in importance: t[5] = importance[t[0]]
    fleet = fleet or REAL_FLEET
    V, D = len(secs), len(REAL_BASES)
    names = [t[0] for t in secs]; A = np.array([t[2] for t in secs], float)
    lat = [t[3] for t in secs] + [b[1][0] for b in REAL_BASES]; lon = [t[4] for t in secs] + [b[1][1] for b in REAL_BASES]
    x, y = _aeqd(lat, lon, float(np.mean(lat)), float(np.mean(lon)))
    P = np.where(A <= PMIN_AREA_KM2, pmin_small, pmin_large)
    I = np.array([t[5] for t in secs], float); score = np.round(25.0 * (I / 5.0) * P)
    size_eff = A * (-np.log(1 - P)) / 200.0
    bnames = [b[0] for b in REAL_BASES]
    air = []
    for cls, base in fleet:
        c = REAL_CLASSES[cls]; air.append(Aircraft(min(c['T'], CREW_LIMIT_MIN), c['v'], c['rho'], bnames.index(base)))
    # time windows: start of pass in [sunrise, sunset - shortest pass among classes], local clock utc_offset_h
    o = np.zeros(V); cl = np.zeros(V); tau_min = np.zeros(V)
    for k in range(V):
        sr, ss = sun_times_utc(lat[k], lon[k], doy); sr += utc_offset_h * 60; ss += utc_offset_h * 60
        taus = [A[k] * (-math.log(1 - P[k])) * 60 / (2 * REAL_CLASSES[c]['rho'] * REAL_CLASSES[c]['v']) for c in REAL_CLASSES]
        tau_min[k] = min(taus); o[k] = round(sr); cl[k] = round(ss - min(taus))
    inst = Instance(name, V, D, len(air), air, x[:V].round(2), y[:V].round(2), size_eff.round(4), score, o, cl, x[V:].round(2), y[V:].round(2)).build()
    inst.meta = dict(
        planning_day_of_year=doy, clock=f'UTC{utc_offset_h:+.0f}', crew_limit_min=CREW_LIMIT_MIN, pmin_rule=f'{pmin_small} if area<={PMIN_AREA_KM2:.0f} km2 else {pmin_large}',
        areas=[dict(id=k + 1, sector=names[k], mpa=secs[k][1], lat=float(lat[k]), lon=float(lon[k]), area_km2=round(float(A[k])), importance=int(I[k]), pmin=float(P[k]),
                    score=int(score[k]), tw_open=int(o[k]), tw_close=int(cl[k]),
                    tau_min={cls: round(float(A[k] * (-math.log(1 - P[k])) * 60 / (2 * REAL_CLASSES[cls]['rho'] * REAL_CLASSES[cls]['v'])), 1) for cls in REAL_CLASSES}) for k in range(V)],
        bases=[dict(id=V + k + 1, name=b[0], lat=b[1][0], lon=b[1][1]) for k, b in enumerate(REAL_BASES)],
        aircraft=[dict(idx=r, cls=cls, base=base, endurance_min=REAL_CLASSES[cls]['T'], route_limit_min=min(REAL_CLASSES[cls]['T'], CREW_LIMIT_MIN),
                       speed_kmh=REAL_CLASSES[cls]['v'], rho_km=REAL_CLASSES[cls]['rho']) for r, (cls, base) in enumerate(fleet)],
        projection='azimuthal equidistant, centre = mean of all points', koopman='tau = -A ln(1-Pmin)/(2 rho v)')
    return inst

def real_instance_tables(inst, out_dir):
    """LaTeX tables for Section 6.4: areas (T6a) and fleet/bases (T6b)."""
    import pandas as pd
    A = pd.DataFrame(inst.meta['areas'])
    def hhmm(m): return f'{int(m)//60:02d}:{int(m)%60:02d}'
    rows = [dict(ID=a.id, Sector=a.sector, MPA=a.mpa, Area=f'{a.area_km2:,.0f}', I=a.importance, Pmin=f'{a.pmin:.1f}', Score=a.score,
                 Window=f'{hhmm(a.tw_open)}--{hhmm(a.tw_close)}', **{f'tau {c.split()[0]}': f"{v:.0f}" for c, v in a.tau_min.items()}) for a in A.itertuples()]
    T = pd.DataFrame(rows)
    L = [r'\begin{table*}[!ht]', r'\centering', r'\caption{Real instance: surveillance sectors, scores, daylight windows (local clock) and Koopman surveillance time in minutes for each aircraft class.\label{tab6}}',
         r'\footnotesize\setlength{\tabcolsep}{3pt}', r'\begin{tabular*}{\textwidth}{@{\extracolsep\fill}' + 'rll' + 'r' * (len(T.columns) - 3) + r'@{\extracolsep\fill}}', r'\toprule',
         ' & '.join(r'\textbf{' + c.replace('tau ', r'$\tau$ ').replace('Pmin', r'$P_{\min}$').replace('Area', r'Area (km$^2$)') + '}' for c in T.columns) + r' \\', r'\midrule']
    for _, r in T.iterrows(): L.append(' & '.join(str(v) for v in r.values) + r' \\')
    L += [r'\bottomrule', r'\end{tabular*}', r'\end{table*}']
    with open(os.path.join(out_dir, 'T6a_real_areas.tex'), 'w') as f: f.write('\n'.join(L) + '\n')
    F = pd.DataFrame(inst.meta['aircraft'])
    L = [r'\begin{table}[!ht]', r'\centering', r'\caption{Real instance: fleet for the planning day (public specifications; route time limited by the 8-hour crew-duty rule) and departure bases.\label{tab7}}',
         r'\footnotesize\setlength{\tabcolsep}{3pt}', r'\begin{tabular*}{\columnwidth}{@{\extracolsep\fill}llrrrr@{\extracolsep\fill}}', r'\toprule',
         r'\textbf{Aircraft} & \textbf{Departure base} & \textbf{Endur.} & \textbf{Limit} & \textbf{Speed} & $\rho$ \\', r' & & (min) & (min) & (km/h) & (km) \\', r'\midrule']
    for _, r in F.iterrows(): L.append(f"{r.cls} & {r.base} & {r.endurance_min:.0f} & {r.route_limit_min:.0f} & {r.speed_kmh:.0f} & {r.rho_km:.0f} \\\\")
    L += [r'\bottomrule', r'\end{tabular*}', r'\end{table}']
    with open(os.path.join(out_dir, 'T6b_real_fleet.tex'), 'w') as f: f.write('\n'.join(L) + '\n')
    return T, F

if __name__ == '__main__':
    import sys
    src = sys.argv[1]; out = sys.argv[2]
    man = generate_all(src, out)
    print({k: len(v) for k, v in man.items()})

In [ ]:
%%writefile /content/mbh_v2/code/mbh_experiments.py
"""
mbh_experiments.py -- campaign driver, checkpointing, statistics and LaTeX export for the MBH-TOPTW study.

A run = (campaign, instance, method). Every finished run is appended immediately to <results_dir>/runs.csv
(one row) and its routes to <results_dir>/routes/<campaign>__<instance>__<method>.json, so a Colab session can be
interrupted and resumed: already-finished runs are skipped.
"""
from __future__ import annotations
import os, json, time, math, traceback, shutil
import numpy as np, pandas as pd
from mbh_core import read_instance, solve_milp, run_cg, verify_solution, INF

COLUMNS = ['campaign', 'instance', 'method', 'R', 'V', 'D', 'value', 'bound', 'gap', 'time', 'status', 'proved', 'iterations',
           'n_columns', 'lp_bound', 'time_limit', 'feasible', 'nodes', 'n_vars', 'n_constrs', 'host', 'timestamp', 'model']

# Version of the optimisation models/heuristics. A row written by an earlier version is ignored (and the run is redone).
#   1: initial release (MILP and exact pricing shared the base time variable: routes returning to their departure base
#      were infeasible for Gurobi -> MILP values and CG bounds were those of a restricted problem)
#   2: separate arrival-time variables, bounded departure time (small big-M), TS end-base re-selection on insertion
MODEL_VERSION = 2

class Campaign:
    def __init__(self, results_dir: str, env=None, threads: int = 0, host: str = ''):
        self.dir = results_dir; os.makedirs(os.path.join(results_dir, 'routes'), exist_ok=True)
        self.csv = os.path.join(results_dir, 'runs.csv'); self.env = env; self.threads = threads; self.host = host
        if os.path.exists(self.csv):
            self.df = pd.read_csv(self.csv)
            missing = [c for c in COLUMNS if c not in self.df.columns]
            if missing:      # file written by an earlier version: keep a copy, add the new columns, rewrite the header
                bak = self.csv + '.v1.bak'
                if not os.path.exists(bak): shutil.copy(self.csv, bak)
                for c in missing: self.df[c] = np.nan
                self.df = self.df.reindex(columns=COLUMNS); self.df.to_csv(self.csv, index=False)
        else:
            self.df = pd.DataFrame(columns=COLUMNS); self.df.to_csv(self.csv, index=False)
    def _valid(self, d):
        """rows usable by the current code: no error, written by the current model version, and (CG) a terminal status"""
        ok = ~d.status.astype(str).str.startswith('error')
        ok &= (pd.to_numeric(d.model, errors='coerce') == MODEL_VERSION)
        ok &= (d.method != 'CG') | self._cg_ok(d)
        return ok
    def done(self, campaign, instance, method):
        """finished without error by the current version AND its routes file exists (a run interrupted while saving is redone)"""
        d = self.df
        hit = bool((self._valid(d) & (d.campaign == campaign) & (d.instance == instance) & (d.method == method)).any())
        return hit and os.path.exists(self._routes_path(campaign, instance, method))
    def clean(self):
        """valid rows; if a run was repeated, the last one counts"""
        d = self.df[self._valid(self.df)].copy()
        return d.drop_duplicates(['campaign', 'instance', 'method'], keep='last').copy()
    def stale(self):
        """rows of earlier versions or with errors (kept in runs.csv for traceability, never used in tables)"""
        return self.df[~self._valid(self.df)].copy()
    @staticmethod
    def _cg_ok(d):
        """CG rows that count as finished: converged (pricing proved), stopped (time limit), unproved (valid Lagrangian
        bound, pricing not proved), no_bound (no valid bound could be computed)."""
        return d.status.astype(str).isin(['converged', 'stopped', 'unproved', 'no_bound'])
    @staticmethod
    def _json_default(o):
        import numpy as _np
        if isinstance(o, (_np.integer,)): return int(o)
        if isinstance(o, (_np.floating,)): return float(o)
        if isinstance(o, (_np.bool_,)): return bool(o)
        if isinstance(o, (_np.ndarray,)): return o.tolist()
        return str(o)
    def _routes_path(self, campaign, instance, method):
        return os.path.join(self.dir, 'routes', f"{campaign}__{instance}__{method}.json")
    def _append(self, row: dict, routes):
        row = {k: row.get(k, None) for k in COLUMNS}
        with open(self._routes_path(row['campaign'], row['instance'], row['method']), 'w') as f:   # routes first: a failure leaves no CSV row
            json.dump(routes, f, default=self._json_default)
        pd.DataFrame([row]).to_csv(self.csv, mode='a', header=False, index=False)
        self.df = pd.read_csv(self.csv)
    def run(self, campaign: str, path: str, method: str, time_limit: float, sp_time: float = 60.0, final_mip_time: float = 600.0,
            verbose: bool = False, force: bool = False):
        name = os.path.basename(path)
        if not force and self.done(campaign, name, method):
            return None
        inst = read_instance(path, name)
        t0 = time.time(); row = dict(campaign=campaign, instance=name, method=method, R=inst.R, V=inst.V, D=inst.D, time_limit=time_limit,
                                     host=self.host, timestamp=time.strftime('%Y-%m-%d %H:%M:%S'), model=MODEL_VERSION)
        try:
            if method == 'MILP':
                res = solve_milp(inst, time_limit=time_limit, env=self.env, threads=self.threads, verbose=verbose)
                ok, val = verify_solution(inst, res['routes'])
                row.update(value=res['value'], bound=res['bound'], gap=res['gap'], time=res['time'], status=res['status'], proved=(res['status'] == 'optimal'),
                           feasible=ok and abs(val - res['value']) < 1e-6, nodes=res['nodes'], n_vars=res['vars'], n_constrs=res['constrs'])
                routes = res['routes']
            else:
                mode = 'CG' if method == 'CG' else 'CG-TS'
                res = run_cg(inst, mode=mode, time_limit=time_limit, env=self.env, threads=self.threads, sp_time=sp_time, verbose=verbose,
                             final_mip_time=final_mip_time)
                ok, val = verify_solution(inst, res.routes)
                has_bound = (res.lagr_bound == res.lagr_bound) and res.lagr_bound < INF
                if mode == 'CG':   # converged: pricing proved; stopped: time limit; unproved: valid Lagrangian bound, pricing not proved
                    status = 'converged' if res.proved else ('stopped' if res.hit_time_limit else ('unproved' if has_bound else 'no_bound'))
                else:              # stopped: time limit; heuristic_stop: TS found no further column after two intensifications
                    status = 'stopped' if res.hit_time_limit else 'heuristic_stop'
                if has_bound and res.integer_value > res.lagr_bound + 1e-6:     # must never happen: a feasible value above a valid bound
                    status = f'error:value {res.integer_value} > bound {res.lagr_bound:.3f}'
                row.update(value=res.integer_value, bound=(res.lagr_bound if has_bound else None),
                           gap=(res.gap if res.gap == res.gap else None), time=res.time_total,
                           status=status,
                           proved=res.proved, iterations=res.iterations, n_columns=res.n_columns, lp_bound=res.lp_bound,
                           feasible=ok and abs(val - res.integer_value) < 1e-6)
                routes = res.routes
        except Exception as e:
            row.update(status='error:' + str(e)[:80]); routes = {'error': traceback.format_exc()}
        self._append(row, routes)
        return row

# ------------------------------------------------------------------------------------------
#  statistics (scipy if available; exact/normal fallbacks otherwise)
# ------------------------------------------------------------------------------------------
def wilcoxon_paired(a, b, alternative='greater'):
    d = np.asarray(a, float) - np.asarray(b, float); d = d[np.abs(d) > 1e-12]; n = len(d)
    if n == 0: return 0.0, 1.0, 0
    try:
        from scipy import stats
        r = stats.wilcoxon(d, alternative=alternative); return float(r.statistic), float(r.pvalue), n
    except Exception:
        pass
    ranks = pd.Series(np.abs(d)).rank().values; Wp = ranks[d > 0].sum(); Wm = ranks[d < 0].sum()
    mu = n*(n+1)/4; sd = math.sqrt(n*(n+1)*(2*n+1)/24); z = (min(Wp, Wm) - mu) / sd
    p2 = 2*(1-0.5*(1+math.erf(abs(z)/math.sqrt(2))))
    if alternative == 'two-sided': return float(Wp), float(min(1, p2)), n
    return float(Wp), float(p2/2 if (Wp > Wm) == (alternative == 'greater') else 1-p2/2), n

def friedman_nemenyi(M: np.ndarray, names, higher_is_better=True):
    n, k = M.shape
    R = np.apply_along_axis(lambda r: pd.Series(-r if higher_is_better else r).rank().values, 1, M).mean(0)
    chi2 = 12*n/(k*(k+1))*(np.sum(R**2) - k*(k+1)**2/4)
    try:
        from scipy import stats; p = float(stats.chi2.sf(chi2, k-1))
    except Exception:
        p = float(math.exp(-chi2/2)) if k == 3 else float('nan')
    q = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728}.get(k, 2.728)
    CD = q*math.sqrt(k*(k+1)/(6*n))
    return dict(chi2=float(chi2), p=p, avg_rank={nm: round(float(r), 3) for nm, r in zip(names, R)}, CD=float(CD), n=int(n))

# ------------------------------------------------------------------------------------------
#  tables
# ------------------------------------------------------------------------------------------
def to_latex(df: pd.DataFrame, path: str, caption: str, label: str, fmt=None, colfmt=None, note=None, wide=True):
    fmt = fmt or {}
    cols = list(df.columns); env = 'table*' if wide else 'table'
    width = r'\textwidth' if wide else r'\columnwidth'
    colfmt = colfmt or ('l' + 'r'*(len(cols)-1))
    L = [f'\\begin{{{env}}}[!ht]', r'\centering', f'\\caption{{{caption}\\label{{{label}}}}}', r'\small',
         f'\\begin{{tabular*}}{{{width}}}{{@{{\\extracolsep\\fill}}{colfmt}@{{\\extracolsep\\fill}}}}', r'\toprule',
         ' & '.join(r'\textbf{' + str(c).replace('_', r'\_').replace('%', r'\%') + '}' for c in cols) + r' \\', r'\midrule']
    for _, row in df.iterrows():
        cells = []
        for c in cols:
            v = row[c]
            if v is None or (isinstance(v, float) and v != v): cells.append('--')
            elif c in fmt: cells.append(fmt[c].format(v))
            elif isinstance(v, float): cells.append(f'{v:.2f}')
            else: cells.append(str(v).replace('_', r'\_').replace('%', r'\%'))
        L.append(' & '.join(cells) + r' \\')
    L += [r'\bottomrule', r'\end{tabular*}']
    if note: L += [r'\begin{tablenotes}', r'\item {\it Note:} ' + note, r'\end{tablenotes}']
    L += [f'\\end{{{env}}}']
    with open(path, 'w') as f: f.write('\n'.join(L) + '\n')

def benchmark_table(df: pd.DataFrame, campaign='benchmark'):
    """One row per instance with MILP / CG / CG-TS columns, gaps as in the paper."""
    d = df[(df.campaign == campaign) & ~df.status.astype(str).str.startswith('error')]
    rows = []
    for inst, g in d.groupby('instance', sort=True):
        r = dict(instance=inst)
        m = g[g.method == 'MILP'].tail(1); c = g[g.method == 'CG'].tail(1); t = g[g.method == 'CG-TS'].tail(1)
        ub_candidates = []
        if len(m) and pd.notna(m.value.iloc[0]):
            r.update(milp_value=float(m.value.iloc[0]), milp_time=float(m.time.iloc[0]), milp_gap=float(m.gap.iloc[0]), milp_bound=float(m.bound.iloc[0]),
                     milp_status=str(m.status.iloc[0]))
        if len(m) and pd.notna(m.bound.iloc[0]): ub_candidates.append(float(m.bound.iloc[0]))
        if len(c):
            r.update(cg_value=float(c.value.iloc[0]), cg_time=float(c.time.iloc[0]), cg_ub=(float(c.bound.iloc[0]) if pd.notna(c.bound.iloc[0]) else np.nan),
                     cg_lp=float(c.lp_bound.iloc[0]), cg_proved=bool(c.proved.iloc[0]))
            if pd.notna(c.bound.iloc[0]): ub_candidates.append(float(c.bound.iloc[0]))
        if len(t): r.update(ts_value=float(t.value.iloc[0]), ts_time=float(t.time.iloc[0]))
        best_ub = min(ub_candidates) if ub_candidates else np.nan
        vals = [r.get(k) for k in ('milp_value', 'cg_value', 'ts_value') if r.get(k) is not None]
        best_known = max(vals) if vals else np.nan
        r['best_ub'] = best_ub; r['best_known'] = best_known
        for k, lab in (('cg_value', 'cg_gap'), ('ts_value', 'ts_gap')):
            r[lab] = (best_ub - r[k]) / best_ub * 100 if (k in r and best_ub == best_ub and best_ub > 0) else np.nan
        rows.append(r)
    return pd.DataFrame(rows)

def random_table(df: pd.DataFrame, campaign='random'):
    """Scenario-level means over replications: time (mean, sd) per method and gaps to the best upper bound."""
    d = df[(df.campaign == campaign) & ~df.status.astype(str).str.startswith('error')].copy()
    d['scenario'] = d.instance.str.extract(r'^(rnd\d\d)_')[0]
    # best UB per instance = min over MILP bound and CG lagrangian bound
    ub = d[d.method.isin(['MILP', 'CG'])].groupby('instance').bound.min()
    d['best_ub'] = d.instance.map(ub)
    d['gap_ub'] = (d.best_ub - d.value) / d.best_ub * 100
    rows = []
    for sc, g in d.groupby('scenario', sort=True):
        r = dict(scenario=sc, R=int(g.R.iloc[0]), V=int(g.V.iloc[0]), D=int(g.D.iloc[0]), n=int(g[g.method == 'MILP'].instance.nunique()))
        for mth, pre in (('MILP', 'milp'), ('CG', 'cg'), ('CG-TS', 'ts')):
            h = g[g.method == mth]
            if len(h):
                r[f'{pre}_time'] = h.time.mean(); r[f'{pre}_sd'] = h.time.std(ddof=1) if len(h) > 1 else 0.0
                r[f'{pre}_gap'] = (h.gap.mean() if mth == 'MILP' else h.gap_ub.mean())
                r[f'{pre}_value'] = h.value.mean()
        rows.append(r)
    return pd.DataFrame(rows), d

def paired_stats(d: pd.DataFrame, col='time'):
    """Wilcoxon (one-sided) and Friedman over instances having all three methods."""
    P = d.pivot_table(index='instance', columns='method', values=col, aggfunc='last').dropna()
    out = {'n': int(len(P))}
    if len(P) < 5: return out
    out['wilcoxon_CG_faster_than_MILP'] = wilcoxon_paired(P['MILP'], P['CG'], 'greater')[1]
    out['wilcoxon_TS_faster_than_CG'] = wilcoxon_paired(P['CG'], P['CG-TS'], 'greater')[1]
    out['friedman'] = friedman_nemenyi(P[['MILP', 'CG', 'CG-TS']].values, ['MILP', 'CG', 'CG-TS'], higher_is_better=(col != 'time'))
    return out

In [ ]:
%%writefile /content/mbh_v2/code/mbh_enumerate.py
"""mbh_enumerate.py -- complete enumeration of the optimal plans of a small instance (Chilean case study, Section 7).

A plan gives every aircraft a set of areas, possibly empty, and a landing base, such that some visiting order of the set
is a feasible route from the departure base of the aircraft to that landing base. Plans that differ only in the visiting
order within a route are counted once; the flight time of a route is the shortest over its feasible visiting orders.
An aircraft with an empty set stays at its departure base (the empty route is counted once, not once per base).
"""
from __future__ import annotations
from functools import lru_cache
from mbh_core import enumerate_routes, route_score


def route_options(inst, max_len: int = 8):
    """per aircraft: list of (area bitmask, score, shortest duration, start, end, route), one per (set, landing base)"""
    out = []
    for r in range(inst.R):
        g = {}
        for (s, e, rt, dur) in enumerate_routes(inst, r, max_len):
            if not rt and e != s:       # an empty route is the aircraft staying at its base: counted once
                continue
            m = 0
            for i in rt:
                m |= 1 << i
            if (m, e) not in g or dur < g[(m, e)][2]:
                g[(m, e)] = (m, float(route_score(inst, rt)), float(dur), s, e, list(rt))
        out.append(sorted(g.values(), key=lambda t: (t[0], t[4])))
    return out


def optimal_plans(inst, max_len: int = 8, tol: float = 1e-9):
    """(optimum, plans); a plan is a list, one entry per aircraft, of (mask, score, duration, start, end, route)"""
    O = route_options(inst, max_len)
    R = inst.R

    @lru_cache(maxsize=None)
    def best(r, used):
        if r == R:
            return 0.0
        return max(sc + best(r + 1, used | m) for (m, sc, *_x) in O[r] if not m & used)

    opt = best(0, 0)
    plans = []

    def rec(r, used, acc, chosen):
        if r == R:
            plans.append(list(chosen))
            return
        for o in O[r]:
            m, sc = o[0], o[1]
            if m & used:
                continue
            if acc + sc + best(r + 1, used | m) >= opt - tol:
                chosen.append(o)
                rec(r + 1, used | m, acc + sc, chosen)
                chosen.pop()

    rec(0, 0, 0.0, [])
    return opt, plans


def best_with_k_aircraft(inst, k: int, max_len: int = 8):
    """highest score attainable when at most k aircraft fly"""
    O = route_options(inst, max_len)
    R = inst.R

    @lru_cache(maxsize=None)
    def best(r, used, left):
        if r == R:
            return 0.0
        v = best(r + 1, used, left)                     # aircraft r stays on the ground
        if left > 0:
            for (m, sc, dur, *_x) in O[r]:
                if dur > 0 and not (m & used):
                    v = max(v, sc + best(r + 1, used | m, left - 1))
        return v

    return best(0, 0, k)


def summary(inst, max_len: int = 8):
    """optimum, number of optimal plans, plans by number of flying aircraft, best score with one aircraft fewer than the
    smallest optimal fleet, range of total flight time among optimal plans, and the optimal plan of least flight time"""
    opt, plans = optimal_plans(inst, max_len)
    n_fly = [sum(1 for o in pl if o[2] > 0) for pl in plans]
    flight = [sum(o[2] for o in pl) for pl in plans]
    k_min = min(n_fly)
    j = min(range(len(plans)), key=lambda q: (flight[q], n_fly[q]))
    return dict(optimum=opt, total_score=float(inst.score.sum()), n_plans=len(plans),
                plans_by_flying_aircraft={k: n_fly.count(k) for k in sorted(set(n_fly))},
                best_with_one_fewer=best_with_k_aircraft(inst, k_min - 1, max_len) if k_min > 1 else 0.0,
                flight_min=min(flight), flight_max=max(flight), min_flight_plan=plans[j]), plans


def describe(inst, plan):
    names = [a['sector'] for a in inst.meta['areas']]
    bases = [b['name'] for b in inst.meta['bases']]
    lines = []
    for r, (m, sc, dur, s, e, rt) in enumerate(plan):
        cls = inst.meta['aircraft'][r]['cls']
        if dur == 0:
            lines.append(f'{cls:16s} {bases[s]}: on the ground')
        else:
            lines.append(f'{cls:16s} {bases[s]} -> ' + ' -> '.join(names[i] for i in rt) + f' -> {bases[e]}   score {sc:.0f}, {dur:.0f} min')
    return '\n'.join(lines)

## 2. Drive, Gurobi e importaciones

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/mbh_toptw'
OUTD = BASE + '/caso_v2'; os.makedirs(OUTD, exist_ok=True)
if not os.path.exists(LIC_PATH) or os.path.getsize(LIC_PATH) < 20 or 'None' in open(LIC_PATH).read():
    assert os.path.exists(BASE + '/gurobi.lic'), 'No hay licencia: usa Colab Secrets o sube gurobi.lic a MyDrive/mbh_toptw/'
    shutil.copy(BASE + '/gurobi.lic', LIC_PATH)

CODE = '/content/mbh_v2/code'
sys.path.insert(0, CODE)
for m in ['mbh_core', 'mbh_instances', 'mbh_experiments', 'mbh_enumerate']:
    sys.modules.pop(m, None)
from mbh_core import read_instance, write_instance, solve_milp, run_cg, verify_solution
from mbh_experiments import Campaign, MODEL_VERSION
import mbh_instances, mbh_enumerate

import gurobipy as gp
from gurobipy import GRB
ENV = gp.Env(); ENV.setParam('OutputFlag', 0)
m = gp.Model(env=ENV); x = m.addVars(5000, ub=1.0); m.setObjective(x.sum(), GRB.MAXIMIZE); m.optimize()
print('Gurobi', gp.gurobi.version(), '| licencia completa:', m.Status == GRB.OPTIMAL); m.dispose()
HOST = f"colab-{os.cpu_count()}cpu"; print('host', HOST)

## 3. Instancia v2 y enumeración completa de los planes óptimos (sin Gurobi)

In [ ]:
WORK = '/content/mbh_v2/out'; shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK + '/instances/real', exist_ok=True); os.makedirs(WORK + '/tables', exist_ok=True)
inst = mbh_instances.build_real_instance(name='real_chile')
PATH = WORK + '/instances/real/real_chile'
write_instance(inst, PATH); json.dump(inst.meta, open(PATH + '_meta.json', 'w'), indent=1)
T, F = mbh_instances.real_instance_tables(inst, WORK + '/tables')
print(T.to_string(index=False)); print(); print(F.to_string(index=False))

t0 = time.time(); S, plans = mbh_enumerate.summary(inst); t_enum = time.time() - t0
names = [a['sector'] for a in inst.meta['areas']]; bnames = [b['name'] for b in inst.meta['bases']]
ENUM = dict(optimum=S['optimum'], total_score=S['total_score'], n_plans=S['n_plans'],
            plans_by_flying_aircraft=S['plans_by_flying_aircraft'], best_with_one_fewer=S['best_with_one_fewer'],
            flight_min=round(S['flight_min'], 2), flight_max=round(S['flight_max'], 2), seconds=round(t_enum, 2),
            plans=[[dict(r=r, start=bnames[o[3]], end=bnames[o[4]], route=[names[i] for i in o[5]], score=o[1], dur=round(o[2], 2))
                    for r, o in enumerate(pl)] for pl in plans])
json.dump(ENUM, open(WORK + '/enumeration_real_chile.json', 'w'), indent=1)
print(f"\noptimo {S['optimum']:.0f} de {S['total_score']:.0f} | planes optimos {S['n_plans']} | por aeronaves que vuelan {S['plans_by_flying_aircraft']}"
      f" | mejor con una aeronave menos {S['best_with_one_fewer']:.0f} | vuelo {S['flight_min']:.0f}-{S['flight_max']:.0f} min ({t_enum:.1f} s)")
print('plan optimo de menor tiempo de vuelo:'); print(mbh_enumerate.describe(inst, S['min_flight_plan']))

## 4. MILP, CG y CG-TS (mismos parámetros de la campaña del paper)

In [ ]:
TL, SP_TIME, FINAL_MIP = 3600.0, 60.0, 600.0
RES = Campaign(WORK + '/results', env=ENV, threads=0, host=HOST)
rows = {}
for mth in ['CG-TS', 'CG', 'MILP']:
    row = RES.run('real', PATH, mth, TL, sp_time=SP_TIME, final_mip_time=FINAL_MIP, force=True)
    rows[mth] = row
    print(f"{mth:6s} value={row.get('value')} bound={row.get('bound')} lp_bound={row.get('lp_bound')} time={row.get('time')} "
          f"status={row.get('status')} proved={row.get('proved')} feasible={row.get('feasible')}", flush=True)

acl = [a['cls'] for a in inst.meta['aircraft']]
for mth in ['MILP', 'CG', 'CG-TS']:
    rts = json.load(open(f"{WORK}/results/routes/real__real_chile__{mth}.json"))
    print(f'\n{mth}:')
    for rt in rts:
        if isinstance(rt, dict) and 'route' in rt and rt['route']:
            print(f"  {acl[rt['r']]:16s}: {bnames[rt['start']]} -> " + ' -> '.join(names[k] for k in rt['route'])
                  + f" -> {bnames[rt['end']]}  (score {rt['score']:.0f}, {rt['dur']:.0f} min)")

opt = S['optimum']
assert all(r.get('feasible') for r in rows.values()), 'alguna solucion no es factible: avisame'
assert rows['MILP']['status'] != 'optimal' or abs(rows['MILP']['value'] - opt) < 1e-6, 'MILP optimo distinto de la enumeracion: avisame'
assert rows['CG'].get('bound') is None or rows['CG']['bound'] >= opt - 1e-6, 'cota CG menor que el optimo: avisame'
print('\nchequeos OK')

## 5. Resumen y descarga
Sube el zip al chat. Si la descarga no funciona, copia y pega el texto que imprime esta celda.

In [ ]:
L = []
L.append(f"MBH-TOPTW caso v2 | {time.strftime('%Y-%m-%d %H:%M')} | {HOST} | Gurobi {gp.gurobi.version()} | model v{MODEL_VERSION}")
L.append(f"enumeracion: optimo {S['optimum']:.0f}/{S['total_score']:.0f}, planes {S['n_plans']}, por aeronaves {S['plans_by_flying_aircraft']}, "
         f"una menos {S['best_with_one_fewer']:.0f}, vuelo {S['flight_min']:.1f}-{S['flight_max']:.1f}")
d = pd.read_csv(WORK + '/results/runs.csv')
L.append(d[['method', 'value', 'bound', 'gap', 'time', 'status', 'proved', 'iterations', 'n_columns', 'lp_bound', 'nodes', 'n_vars', 'n_constrs', 'feasible']].to_string(index=False))
for mth in ['MILP', 'CG', 'CG-TS']:
    rts = json.load(open(f"{WORK}/results/routes/real__real_chile__{mth}.json"))
    L.append(mth + ': ' + json.dumps([dict(r=rt['r'], s=rt['start'], e=rt['end'], route=rt['route'], score=rt['score'], dur=round(rt['dur'], 1))
                                      for rt in rts if isinstance(rt, dict) and 'route' in rt]))
TXT = '\n'.join(L); open(WORK + '/RESUMEN.txt', 'w').write(TXT); print(TXT)

ZIP = shutil.make_archive('/content/mbh_caso_v2_resultados', 'zip', WORK)
shutil.copy(ZIP, OUTD); shutil.copytree(WORK, OUTD + '/out', dirs_exist_ok=True)
print('\ncopia en', OUTD)
from google.colab import files
files.download(ZIP)